# 03 · Join Sofascore + Capology — Turkey Süper Lig 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de Süper Lig turca**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_turkey_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_turkey_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  632 jugadores | 116 columnas
Capology:   731 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   basaksehir fk
   besiktas jk
   fatih karagumruk
   gaziantep fk
   kasmpasa
   mke ankaragucu

En Capology pero no en Sofascore:
   ankaragucu
   basaksehir
   besiktas
   gaziantep bb
   karagumrukspor
   kasimpasa


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'ankaragucu':'mke ankaragucu',
            'basaksehir':'basaksehir fk',
            'besiktas':'besiktas jk',
            'gaziantep bb':'gaziantep fk',
            'karagumrukspor':'fatih karagumruk',
            'kasimpasa':'kasmpasa'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 542/632 (85.8%)
Sin emparejar: 90


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          22
Revisión media    (0.75 ≤ score < 0.90):   25
Revisión estricta (0.50 ≤ score < 0.75):   31
Revisión muy est. (score < 0.50):           12


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
20,Bakhtiyar Zaynutdinov,Beşiktaş JK,baktiyar zaynutdinov,0.976
51,Dimitris Kolovetsios,Kayserispor,dimitrios kolovetsios,0.976
72,Ahmet Kağan Malatyalı,Kayserispor,ahmet kagan malatyali,0.976
26,Husniddin Alikulov,Çaykur Rizespor,khusniddin alikulov,0.973
52,Talha Sariarslan,Kayserispor,talha sararslan,0.968
65,Tunahan Samdanli,İstanbulspor,tunahan samdanl,0.968
89,Alperen Kuyubaşı,MKE Ankaragücü,alperen kuyubasi,0.968
27,Şener Özbayraklı,Başakşehir FK,sener ozbayrakli,0.968
13,Jakub Kałuziński,Antalyaspor,jakub kaluzinski,0.968
12,Rafał Gikiewicz,MKE Ankaragücü,rafal gikiewicz,0.966


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
75,Muhammet Ali Özbaskıcı,Samsunspor,muhammet ozbaskc,0.889
8,Bahadır Han Güngördü,MKE Ankaragücü,bahadr gungordu,0.882
43,Trazie Thomas Zai,Kasımpaşa,trazie thomas,0.867
1,Poyraz Efe Yıldırım,Trabzonspor,poyraz yldrm,0.857
88,Hasan Yesilyurt,Kasımpaşa,hasan emre yesilyurt,0.857
71,Muhammed Arıkan,Kayserispor,muhammed eren arkan,0.848
0,Đorđe Nikolić,Sivasspor,djordje nikolic,0.846
53,Baran Ali Gezek,Kayserispor,baran gezek,0.846
14,Engin Aksoy,Hatayspor,engin can aksoy,0.846
81,Furkan Onur Akyüz,Fenerbahçe,furkan akyuz,0.828


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 25 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
46,Mehmet Eray Özbek,Kayserispor,eray ozbek,0.741
16,Eren Elmalı,Trabzonspor,evren eren elmali,0.741
58,Tuncer Duhan Aksu,Kasımpaşa,duhan aksu,0.741
78,Maximiliano Gómez,Trabzonspor,maxi gomez,0.741
57,Yakup Arda Kılıç,Beşiktaş JK,arda klc,0.727
87,Muhammet Berkay Tekke,İstanbulspor,berkay tekke,0.727
5,Barış Alper Yılmaz,Galatasaray,baris yilmaz,0.714
4,Mahmoud Trézéguet,Trabzonspor,trezeguet,0.692
21,Mutassim Al-Musrati,Beşiktaş JK,al musrati,0.690
84,Bunyamin Cetinkaya,Kasımpaşa,erdem cetinkaya,0.667


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mehmet eray ozbek',
                    'eren elmal',
                    'tuncer duhan aksu',
                    'maximiliano gomez',
                    'yakup arda klc',
                    'muhammet berkay tekke',
                    'bars alper ylmaz',
                    'mahmoud trezeguet',
                    'mutassim al musrati',
                    'bunyamin cetinkaya',
                    'oscar pinchi',
                    'seyfettin anl yasar',
                    'guilherme haubert sitya'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 13


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
80,Deniz Eren Dönmezer,Adana Demirspor,stiven mendoza,0.485
38,Konrad Michalak,Konyaspor,alexandru cicaldau,0.485
15,Marius Mouandilmadji,Samsunspor,marc bola,0.483
50,Douglas Tanque,Samsunspor,mustafa tan,0.480
76,Altay Bayındır,Fenerbahçe,samet akaydin,0.480
73,Musa Çağıran,Hatayspor,nazm ozcan,0.476
34,Souza,Başakşehir FK,ousseynou ba,0.471
77,Ahmed Touba,Başakşehir FK,mehdi abeid,0.455
37,Giorgi Beridze,MKE Ankaragücü,tolga cigerci,0.444
62,Georgios Tzavellas,Pendikspor,leandro kappel,0.438


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 602/632 (95.3%)
Sin salario:     30


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 30


,player,team,minutesPlayed,appearances,goals,assists
0,Cherif Ndiaye,Adana Demirspor,167,2,1,1
1,Deniz Eren Dönmezer,Adana Demirspor,1,1,0,0
2,Samet Duyur,Adana Demirspor,1,1,0,0
3,Fredy,Antalyaspor,141,3,0,0
4,Souza,Başakşehir FK,407,11,0,1
5,Ahmed Touba,Başakşehir FK,57,1,0,0
6,Fahri Ay,Beşiktaş JK,91,2,0,0
7,Azad Demir,Beşiktaş JK,13,1,0,0
8,Adem Ljajić,Fatih Karagümrük,83,2,0,0
9,Nicholas Anwan,Fatih Karagümrük,26,3,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Adana Demirspor  —  SF sin salario:


,player,minutesPlayed
0,Cherif Ndiaye,167
1,Deniz Eren Dönmezer,1
2,Samet Duyur,1


  CG plantilla completa:


,player,player_norm
0,Abat Aymbetov,abat aymbetov
1,Abdulsamet Burak,abdulsamet burak
2,Abdurrahim Dursun,abdurrahim dursun
3,Amir Feratovic,amir feratovic
4,Andreaw Gravillon,andreaw gravillon
5,Arbër Zeneli,arber zeneli
6,Badou Ndiaye,badou ndiaye
7,Benjamin Stambouli,benjamin stambouli
8,Breyton Fougeu,breyton fougeu
9,Burhan Ersoy,burhan ersoy



  Antalyaspor  —  SF sin salario:


...

,player,player_norm
0,Abdülkadir Ömür,abdulkadir omur
1,Anastasios Bakasetas,anastasios bakasetas
2,Arif Boşluk,arif bosluk
3,Batista Mendy,batista mendy
4,Berat Özdemir,berat ozdemir
5,Dimitrios Kourbelis,dimitrios kourbelis
6,Edin Visca,edin visca
7,Enis Bardhi,enis bardhi
8,Enis Destan,enis destan
9,Evren Eren Elmali,evren eren elmali



  İstanbulspor  —  SF sin salario:


,player,minutesPlayed
0,Baran Vardar,112
1,Demir Mermerci,16
2,Emir Mustafa Vuruşaner,10
3,Eren Arda Şan,52


  CG plantilla completa:


,player,player_norm
0,Alassane Ndao,alassane ndao
1,Ali Yaşar,ali yasar
2,Alp Arda,alp arda
3,Alperen Çaylak,alperen caylak
4,Bartu Kırtaş,bartu krtas
5,Berkay Tekke,berkay tekke
6,David Jensen,david jensen
7,David Sambissa,david sambissa
8,Demeaco Duhaney,demeaco duhaney
9,Djakaridja Junior Traoré,djakaridja junior traore


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('marius mouandilmadji', 'samsunspor')    : ('marius', 'samsunspor')
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: marius mouandilmadji (samsunspor) → marius (samsunspor)

Tras matches manuales: 603/632 (95.4%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_turkey_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_turkey_2324.csv
   Jugadores totales:  632
   Con salario:        603
   Sin salario (NaN):  29
   Columnas:           121
